# NeoNatal Watch AI — Phase 1: Data Exploration

> **DEMO / SYNTHETIC DATA — NOT FOR CLINICAL USE**
>
> This notebook explores the synthetic dataset and documents findings.
> All data is simulated and does NOT represent real patients.

---

## Goals of this Notebook

1. Load and inspect the synthetic NICU vital sign dataset
2. Understand the distribution of each vital sign
3. Visualize normal vs abnormal event patterns
4. Examine class imbalance
5. Identify preprocessing needs
6. Document observations for Phase 2 planning


In [ ]:
# Cell 1: Setup and imports
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

print('Imports OK')
print('[WARNING] All data in this notebook is SYNTHETIC - NOT for clinical use')

In [ ]:
# Cell 2: Load synthetic dataset
DATA_PATH = '../data/synthetic/synthetic_nicu_vitals.csv'
df = pd.read_csv(DATA_PATH, parse_dates=['timestamp'])

print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head(5)

In [ ]:
# Cell 3: Basic statistics
print('=== Dataset Overview ===')
print(f'Total records:  {len(df):,}')
print(f'Patients:       {df["patient_id"].nunique()}')
print(f'Date range:     {df["timestamp"].min()} to {df["timestamp"].max()}')
print()
print('=== Event (Label) Distribution ===')
print(df['deterioration_label'].value_counts())
print()
print(f'Event rate: {df["deterioration_label"].mean()*100:.2f}%')
print('NOTE: Low event rate is expected - real NICU deterioration events are rare.')

In [ ]:
# Cell 4: Vital sign statistics
vitals = ['heart_rate', 'spo2', 'respiratory_rate',
          'temperature', 'systolic_bp', 'diastolic_bp']
print('=== Vital Sign Statistics ===')
df[vitals].describe().round(2)

In [ ]:
# Cell 5: Distribution plots (normal vs event)
vitals = ['heart_rate', 'spo2', 'respiratory_rate',
          'temperature', 'systolic_bp', 'diastolic_bp']
vital_titles = {
    'heart_rate':       'Heart Rate (bpm)',
    'spo2':             'SpO2 (%)',
    'respiratory_rate': 'Respiratory Rate (breaths/min)',
    'temperature':      'Temperature (C)',
    'systolic_bp':      'Systolic BP (mmHg)',
    'diastolic_bp':     'Diastolic BP (mmHg)',
}

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Vital Sign Distributions (SYNTHETIC DATA - NOT FOR CLINICAL USE)',
             fontsize=11, color='red', fontweight='bold')

for i, (col, title) in enumerate(vital_titles.items()):
    ax = axes.flatten()[i]
    normal_data = df[df['deterioration_label'] == 0][col]
    event_data  = df[df['deterioration_label'] == 1][col]
    ax.hist(normal_data, bins=50, alpha=0.7, color='steelblue', label='Normal')
    ax.hist(event_data,  bins=20, alpha=0.8, color='crimson',   label='Event')
    ax.set_title(title, fontsize=9)
    ax.set_ylabel('Count')
    ax.legend(fontsize=8)

plt.tight_layout()
os.makedirs('../reports/figures', exist_ok=True)
plt.savefig('../reports/figures/01_vital_sign_distributions.png', dpi=100, bbox_inches='tight')
plt.show()
print('Saved: reports/figures/01_vital_sign_distributions.png')

In [ ]:
# Cell 6: Patient timeline visualization
# We use SYN-004 because it has some events
patient_id = 'SYN-004'
patient = df[df['patient_id'] == patient_id].copy()
print(f'Patient {patient_id}: {len(patient)} rows, '
      f'{patient["deterioration_label"].sum()} event rows')

plot_data = [
    ('heart_rate',       'HR (bpm)',    'steelblue'),
    ('spo2',             'SpO2 (%)',    'green'),
    ('respiratory_rate', 'RR (br/min)', 'orange'),
    ('temperature',      'Temp (C)',    'purple'),
]

fig, axes = plt.subplots(4, 1, figsize=(16, 12), sharex=True)
fig.suptitle(f'Synthetic Patient {patient_id} - 24h Timeline\n(SYNTHETIC DATA - NOT FOR CLINICAL USE)',
             color='red', fontsize=11)

for ax, (col, ylabel, color) in zip(axes, plot_data):
    ax.plot(patient['timestamp'], patient[col], color=color, linewidth=0.4, alpha=0.8)
    event_mask = patient['deterioration_label'] == 1
    if event_mask.any():
        ax.scatter(patient.loc[event_mask, 'timestamp'],
                   patient.loc[event_mask, col],
                   color='red', s=8, zorder=5, label='Simulated Event')
        ax.legend(fontsize=8)
    ax.set_ylabel(ylabel, fontsize=9)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('Time')
plt.tight_layout()
plt.savefig('../reports/figures/02_patient_timeline.png', dpi=100, bbox_inches='tight')
plt.show()
print('Saved: reports/figures/02_patient_timeline.png')

In [ ]:
# Cell 7: Correlation matrix
vitals = ['heart_rate', 'spo2', 'respiratory_rate',
          'temperature', 'systolic_bp', 'diastolic_bp']
corr = df[vitals].corr()

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(vitals)))
ax.set_yticks(range(len(vitals)))
short = [v.replace('_', ' ') for v in vitals]
ax.set_xticklabels(short, rotation=30, fontsize=8)
ax.set_yticklabels(short, fontsize=8)
plt.colorbar(im, ax=ax)
ax.set_title('Vital Sign Correlation (SYNTHETIC DATA)', color='red')

for i in range(len(vitals)):
    for j in range(len(vitals)):
        ax.text(j, i, f'{corr.iloc[i, j]:.2f}', ha='center', va='center', fontsize=8)

plt.tight_layout()
plt.savefig('../reports/figures/03_correlation_matrix.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 8: Missing value analysis
print('=== Missing Value Analysis ===')
vitals = ['heart_rate', 'spo2', 'respiratory_rate',
          'temperature', 'systolic_bp', 'diastolic_bp']
missing = df[vitals].isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
summary = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(summary)
print()
print('NOTE: Synthetic data has 0 missing values by design.')
print('CinC 2019 and MIMIC-III will have significant missing data -> Phase 2 task.')

## Phase 1 Observations Summary

### What We Found:

| Metric | Value | Notes |
|--------|-------|-------|
| Total records | 172,800 | 20 patients x 24h x 10s intervals |
| Patients | 20 | Synthetic, all labeled SYN-001 to SYN-020 |
| HR mean | ~140 bpm | Within preterm neonatal range |
| SpO2 mean | ~96% | Within acceptable preterm range |
| Event rate | ~0.3% | Extreme class imbalance! |
| Missing values | 0 | Synthetic data only |

### Key Finding — Class Imbalance:

> A model that predicts "0" (normal) for every sample would be **99.7% accurate**
> but **completely useless** clinically.
>
> **Solution: Use PR-AUC as primary metric, not accuracy.**
> Also use `scale_pos_weight` in XGBoost and class weighting in deep learning.

### What Phase 2 Must Do:

1. Handle missing values (CinC 2019 has ~60-80% missing per column per patient)
2. Resample from 10s to 1-minute intervals for initial ML
3. Detect and clamp physiological outliers
4. Patient-level train/val/test split (not random row split!)
5. Normalize vital signs to 0-1 scale
6. Create sliding window sequences for CNN-LSTM

---
*Phase 1 Complete. Proceed to Phase 2: Data Preprocessing.*
